# finance / household / flexibility

The essential spending floor and the share of spending that can be cut.

| Term | In This Notebook |
|---|---|
| **Intent** | Split spending into essential, committed, discretionary and irregular, and say how much could be cut in a bad year. Placeholder: eight synthetic categories |
| **Project Flavour** | `static`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` eight synthetic categories with their class and annual spend, until the `bank` table exists; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/flexibility"
MODEL_NAME = "finance_household_flexibility"
TARGETS = [{"kind": "local"}]
TRAINING = "household_flexibility_training_2026-11-05_2026-11-05"
GOLDEN = "household_flexibility_golden"
INPUT = "finance_household_flexibility_in"
OUTPUTS = ["finance_household_flexibility_essential_floor_dollars", "finance_household_flexibility_discretionary_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {
        "category": ["groceries", "utilities", "insurance", "school", "hospitality", "subscriptions", "holidays", "lifestyle"],
        "class": ["essential", "essential", "committed", "committed", "discretionary", "discretionary", "discretionary", "discretionary"],
        "annual_dollars": [20000.0, 8000.0, 6000.0, 6000.0, 10000.0, 2000.0, 8000.0, 40000.0],
    },
)
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the essential floor as the essential plus committed categories of `training` only (R6) and the discretionary share as the rest over the total. The classification itself is the judgement the plan says must be written down.

In [ ]:
fixed = training_pl.filter(training_pl["class"].is_in(["essential", "committed"]))["annual_dollars"].sum()
total = training_pl["annual_dollars"].sum()
CONSTANTS = dict(zip(OUTPUTS, [round(float(fixed), 0), round(float((total - fixed) / total * 100.0), 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [40000.0, 60.0], "the split did not give the floor and the share"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_household_flexibility_essential_floor_dollars", "none", "Annual spending that cannot be cut, essential plus committed, dollars.", datatype="FP64"),
        signature_output("finance_household_flexibility_discretionary_pct", "percent", "Share of annual spending that is discretionary, percent.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic categories with a class, training split",
        [training_path, golden_path],
        "sum of the essential and committed classes against the total",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")